In [ ]:
#Import required libraries

import pandas as pd
import numpy as np
import re

# Used to split data
from sklearn.model_selection import train_test_split

# Used to convert text into TF-IDF features
from sklearn.feature_extraction.text import TfidfVectorizer

# Naive Bayes classifier
from sklearn.naive_bayes import MultinomialNB

# Evaluation metrics
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
# Upload AG News dataset files

from google.colab import files

uploaded = files.upload()

Saving test.csv to test.csv
Saving train.csv to train.csv


In [ ]:
# Load the training dataset

train_df = pd.read_csv("train.csv")

# Display first 5 rows
train_df.head()

,Class Index,Title,Description
0,3,Wall St. Bears Claw Back Into the Black (Reuters),"Reuters - Short-sellers, Wall Street's dwindli..."
1,3,Carlyle Looks Toward Commercial Aerospace (Reu...,Reuters - Private investment firm Carlyle Grou...
2,3,Oil and Economy Cloud Stocks' Outlook (Reuters),Reuters - Soaring crude prices plus worries\ab...
3,3,Iraq Halts Oil Exports from Main Southern Pipe...,Reuters - Authorities have halted oil export\f...
4,3,"Oil prices soar to all-time record, posing new...","AFP - Tearaway world oil prices, toppling reco..."


In [ ]:
# Load the testing dataset

test_df = pd.read_csv("test.csv")

# Display first 5 rows
test_df.head()

,Class Index,Title,Description
0,3,Fears for T N pension after talks,Unions representing workers at Turner Newall...
1,4,The Race is On: Second Private Team Sets Launc...,"SPACE.com - TORONTO, Canada -- A second\team o..."
2,4,Ky. Company Wins Grant to Study Peptides (AP),AP - A company founded by a chemistry research...
3,4,Prediction Unit Helps Forecast Wildfires (AP),AP - It's barely dawn when Mike Fitzpatrick st...
4,4,Calif. Aims to Limit Farm-Related Smog (AP),AP - Southern California's smog-fighting agenc...


In [ ]:
# Check the structure of the datasets

print("Training Dataset Shape:", train_df.shape)
print("Testing Dataset Shape:", test_df.shape)

print("\nTraining Columns:")
print(train_df.columns)

print("\nTesting Columns:")
print(test_df.columns)

Training Dataset Shape: (120000, 3)
Testing Dataset Shape: (7600, 3)

Training Columns:
Index(['Class Index', 'Title', 'Description'], dtype='object')

Testing Columns:
Index(['Class Index', 'Title', 'Description'], dtype='object')


In [ ]:
# Assign column names

train_df.columns = ["class", "title", "description"]
test_df.columns = ["class", "title", "description"]

print(train_df.columns)

Index(['class', 'title', 'description'], dtype='object')


In [ ]:
# Combine title and description into one text column

train_df["text"] = (
    train_df["title"].fillna("") + " " +
    train_df["description"].fillna("")
)

test_df["text"] = (
    test_df["title"].fillna("") + " " +
    test_df["description"].fillna("")
)

# Display the combined text
train_df[["title", "description", "text"]].head()

,title,description,text
0,Wall St. Bears Claw Back Into the Black (Reuters),"Reuters - Short-sellers, Wall Street's dwindli...",Wall St. Bears Claw Back Into the Black (Reute...
1,Carlyle Looks Toward Commercial Aerospace (Reu...,Reuters - Private investment firm Carlyle Grou...,Carlyle Looks Toward Commercial Aerospace (Reu...
2,Oil and Economy Cloud Stocks' Outlook (Reuters),Reuters - Soaring crude prices plus worries\ab...,Oil and Economy Cloud Stocks' Outlook (Reuters...
3,Iraq Halts Oil Exports from Main Southern Pipe...,Reuters - Authorities have halted oil export\f...,Iraq Halts Oil Exports from Main Southern Pipe...
4,"Oil prices soar to all-time record, posing new...","AFP - Tearaway world oil prices, toppling reco...","Oil prices soar to all-time record, posing new..."


In [ ]:
# Check the distribution of news categories

print("Training Class Distribution:")
print(train_df["class"].value_counts())

print("\nTesting Class Distribution:")
print(test_df["class"].value_counts())

Training Class Distribution:
class
3    30000
4    30000
2    30000
1    30000
Name: count, dtype: int64

Testing Class Distribution:
class
3    1900
4    1900
2    1900
1    1900
Name: count, dtype: int64


In [ ]:
# Clean the text data

def clean_text(text):

    # Convert text to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)

    # Remove punctuation and special characters
    text = re.sub(r"[^a-zA-Z\s]", " ", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text


# Apply cleaning to training and testing text
train_df["clean_text"] = train_df["text"].apply(clean_text)
test_df["clean_text"] = test_df["text"].apply(clean_text)

# Display original and cleaned text
train_df[["text", "clean_text"]].head()

,text,clean_text
0,Wall St. Bears Claw Back Into the Black (Reute...,wall st bears claw back into the black reuters...
1,Carlyle Looks Toward Commercial Aerospace (Reu...,carlyle looks toward commercial aerospace reut...
2,Oil and Economy Cloud Stocks' Outlook (Reuters...,oil and economy cloud stocks outlook reuters r...
3,Iraq Halts Oil Exports from Main Southern Pipe...,iraq halts oil exports from main southern pipe...
4,"Oil prices soar to all-time record, posing new...",oil prices soar to all time record posing new ...


In [ ]:
# Separate input text and output labels

X_train_text = train_df["clean_text"]
y_train = train_df["class"]

X_test_text = test_df["clean_text"]
y_test = test_df["class"]

print("Training text:")
print(X_train_text.head())

print("\nTraining labels:")
print(y_train.head())

Training text:
0    wall st bears claw back into the black reuters...
1    carlyle looks toward commercial aerospace reut...
2    oil and economy cloud stocks outlook reuters r...
3    iraq halts oil exports from main southern pipe...
4    oil prices soar to all time record posing new ...
Name: clean_text, dtype: object

Training labels:
0    3
1    3
2    3
3    3
4    3
Name: class, dtype: int64


In [ ]:
# Convert text into TF-IDF numerical features

vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=20000
)

# Learn vocabulary from training data
# and transform training text into TF-IDF features
X_train = vectorizer.fit_transform(X_train_text)

# Transform test data using the same vocabulary
X_test = vectorizer.transform(X_test_text)

print("Training TF-IDF Shape:", X_train.shape)
print("Testing TF-IDF Shape:", X_test.shape)

Training TF-IDF Shape: (120000, 20000)
Testing TF-IDF Shape: (7600, 20000)


In [ ]:
# Create and train the Multinomial Naive Bayes classifier

model = MultinomialNB()

# Train the model
model.fit(X_train, y_train)

print("Multinomial Naive Bayes model trained successfully.")

Multinomial Naive Bayes model trained successfully.


In [ ]:
# Make predictions on the test dataset

y_pred = model.predict(X_test)

# Display first 20 predictions
print("First 20 Predicted Classes:")
print(y_pred[:20])

print("\nFirst 20 Actual Classes:")
print(y_test.values[:20])

First 20 Predicted Classes:
[3 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 3]

First 20 Actual Classes:
[3 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4]


In [ ]:
# Calculate accuracy

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)
print("Accuracy Percentage:", accuracy * 100, "%")

Accuracy: 0.9023684210526316
Accuracy Percentage: 90.23684210526316 %


In [ ]:
# Display classification report

target_names = [
    "World",
    "Sports",
    "Business",
    "Sci/Tech"
]

print("Classification Report:\n")

print(
    classification_report(
        y_test,
        y_pred,
        labels=[1, 2, 3, 4],
        target_names=target_names
    )
)

Classification Report:

              precision    recall  f1-score   support

       World       0.91      0.89      0.90      1900
      Sports       0.95      0.98      0.96      1900
    Business       0.87      0.86      0.86      1900
    Sci/Tech       0.88      0.88      0.88      1900

    accuracy                           0.90      7600
   macro avg       0.90      0.90      0.90      7600
weighted avg       0.90      0.90      0.90      7600



In [ ]:
# Test the trained model with a new news article

new_news = [
    "The football team won the championship after defeating its opponent."
]

# Clean the new news
new_news_clean = [clean_text(new_news[0])]

# Convert the new text into TF-IDF
new_news_tfidf = vectorizer.transform(new_news_clean)

# Predict the class
prediction = model.predict(new_news_tfidf)

# Convert class number into category name
class_names = {
    1: "World",
    2: "Sports",
    3: "Business",
    4: "Sci/Tech"
}

print("News:", new_news[0])
print("Predicted Class:", prediction[0])
print("Predicted Category:", class_names[prediction[0]])

News: The football team won the championship after defeating its opponent.
Predicted Class: 2
Predicted Category: Sports


In [ ]:
# Test multiple new news articles

new_news = [
    "The football team won the championship.",
    "The company announced a major increase in its annual profits.",
    "Scientists developed a new artificial intelligence system.",
    "The president met with world leaders to discuss international relations."
]

# Clean the news
clean_news = [clean_text(news) for news in new_news]

# Convert to TF-IDF
new_news_tfidf = vectorizer.transform(clean_news)

# Predict
predictions = model.predict(new_news_tfidf)

# Display predictions
for news, prediction in zip(new_news, predictions):
    print("News:", news)
    print("Predicted Category:", class_names[prediction])
    print("-" * 60)

News: The football team won the championship.
Predicted Category: Sports
------------------------------------------------------------
News: The company announced a major increase in its annual profits.
Predicted Category: Business
------------------------------------------------------------
News: Scientists developed a new artificial intelligence system.
Predicted Category: Sci/Tech
------------------------------------------------------------
News: The president met with world leaders to discuss international relations.
Predicted Category: World
------------------------------------------------------------
